# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

In [1]:
# trl — бібліотека для донавчання мовних моделей (SFT, DPO тощо),
# peft — реалізація LoRA та інших методів PEFT, bitsandbytes — 4-бітна квантизація для QLoRA
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
# Colab має застарілу бібліотеку torchao, з якою peft не імпортується; для цього демо вона не потрібна
!pip uninstall -q -y torchao

In [2]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [3]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}


In [4]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [5]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [6]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) — на ньому модель вчиться.
- **Валідаційний** (200) — на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) — на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 — на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [7]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'content': 'Classify the customer message. Reply with JSON only.',
  'role': 'system'},
 {'content': 'I have got to locate hte bills from {{Person Name}}',
  'role': 'user'},
 {'content': '{"category": "INVOICE", "intent": "check_invoice"}',
  'role': 'assistant'}]

## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [8]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [9]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [10]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [11]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` — loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` — кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування — 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` — для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` — кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` — змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` — активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше памʼяті.

In [12]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,                  # діалог із цього датасету після шаблону займає близько 100 токенів
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # ефективний розмір батча: 4 × 2 = 8 прикладів
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову памʼять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # Для QLoRA бібліотека TRL переводить ваги адаптера в bf16, який T4 не підтримує.
    # Повертаємо їх у fp32: змішана точність fp16 потребує, щоб параметри, що тренуються, були у fp32
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова памʼять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити памʼять для наступної.

In [13]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

In [14]:
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB

До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (7 outputs left out)

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.506702,0.400654,0.385706,0.912863,4323.000000
20,0.262003,0.117909,0.144082,0.963722,8672.000000
30,0.067879,0.042200,0.062891,0.988512,13032.000000
40,0.039489,0.023353,0.033976,0.991258,17377.000000
50,0.018554,0.009793,0.022544,0.997490,21714.000000
60,0.006624,0.006435,0.016156,0.998490,26070.000000
70,0.011576,0.005410,0.014623,0.998751,30452.000000
80,0.006360,0.004754,0.012458,0.999248,34813.000000
90,0.004497,0.005541,0.012540,0.999255,39116.000000
100,0.014833,0.005529,0.012465,0.999255,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,98.00
"Тривалість, хв",NaN,2.50
"Пікова памʼять GPU, GB",NaN,1.42
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


In [15]:
run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,2.098361,0.382155,0.314433,0.916777,4323.000000
20,0.272323,0.118023,0.170292,0.963533,8672.000000
30,0.070977,0.041607,0.044197,0.987507,13032.000000
40,0.036560,0.019286,0.027792,0.992774,17377.000000
50,0.020546,0.018643,0.025470,0.993011,21714.000000
60,0.010711,0.008823,0.014125,0.997251,26070.000000
70,0.009548,0.007133,0.010058,0.997756,30452.000000
80,0.008964,0.005898,0.009022,0.998500,34813.000000
90,0.012703,0.005122,0.008869,0.998754,39116.000000
100,0.008004,0.004864,0.008790,0.999009,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",62.7,99.30
"Точність intent, %",57.4,96.90
"Тривалість, хв",NaN,2.56
"Пікова памʼять GPU, GB",NaN,3.83
Validation loss,NaN,0.00
"Адаптер, MB",NaN,69.78


In [16]:
run_lora("Qwen/Qwen3-4B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.921858,0.262515,0.257834,0.933732,4323.000000
20,0.178441,0.087518,0.102185,0.970975,8672.000000
30,0.058810,0.020332,0.034030,0.994246,13032.000000
40,0.024267,0.013813,0.019558,0.996760,17377.000000
50,0.011439,0.007307,0.014985,0.997488,21714.000000
60,0.005264,0.003784,0.008959,0.998740,26070.000000
70,0.008181,0.004104,0.007039,0.998740,30452.000000
80,0.006710,0.003833,0.006030,0.998240,34813.000000
90,0.009619,0.003062,0.006149,0.998995,39116.000000
100,0.008354,0.002977,0.006162,0.998995,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",88.1,100.00
"Точність intent, %",72.5,97.60
"Тривалість, хв",NaN,3.81
"Пікова памʼять GPU, GB",NaN,8.75
Validation loss,NaN,0.00
"Адаптер, MB",NaN,132.19


### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [17]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00
"Точність intent, %",25.6,98.00,57.4,96.90,72.5,97.60
"Тривалість, хв",NaN,2.50,NaN,2.56,NaN,3.81
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` — тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` — для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [18]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.737474,0.605169,0.271417,0.882885,4323.000000
20,0.282767,0.056560,0.095230,0.985241,8672.000000
30,0.037163,0.049979,0.014630,0.987977,13032.000000
40,0.038285,0.020181,0.016560,0.993763,17377.000000
50,0.012865,0.011908,0.009553,0.996995,21714.000000
60,0.003235,0.009957,0.007164,0.997997,26070.000000
70,0.006116,0.009358,0.008030,0.997501,30452.000000
80,0.011116,0.012613,0.006138,0.996752,34813.000000
90,0.004606,0.011770,0.004709,0.997001,39116.000000
100,0.021278,0.010938,0.004558,0.997001,43449.000000


📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,Qwen3-0.6B LoRA,Qwen3-0.6B повне
"Чистий JSON, %",100.00,100.00
"Валідний JSON, %",100.00,100.00
"Точність category, %",99.70,99.40
"Точність intent, %",98.00,95.90
"Тривалість, хв",2.50,1.66
"Пікова памʼять GPU, GB",1.42,10.80
Validation loss,0.01,0.01
"Адаптер, MB",40.42,NaN


## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [19]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

README.md                                0.0 MB
adapter_config.json                      0.0 MB
adapter_model.safetensors              132.2 MB
chat_template.jinja                      0.0 MB
tokenizer.json                          11.4 MB
tokenizer_config.json                    0.0 MB
training_args.bin                        0.0 MB

Для порівняння, ваги Qwen3-4B у fp16: ~8 GB


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [20]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)

# Злиту модель можна зберегти як звичайну: model.save_pretrained("qwen3-4b-merged")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM


Перевіримо злиту модель на власних повідомленнях:

In [21]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_period"}

📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "order_confirmed"}

📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}

📩 I guess, you have shipped to a wrong address
🤖 {"category": "SHIPPING", "intent": "shipping_address"}



In [22]:
del model
free_memory()

## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб — найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** — 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** — 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 — формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше — у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [23]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-бітний формат, розрахований на нормально розподілені ваги
    bnb_4bit_compute_dtype=torch.float16,   # обчислення виконуються у fp16
    bnb_4bit_use_double_quant=True,         # квантизуються також масштаби квантизації: ще трохи менше памʼяті
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Памʼять GPU після завантаження: 2.7 GB


In [24]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.882175,0.281270,0.300003,0.930995,4323.000000
20,0.190657,0.109022,0.101652,0.963483,8672.000000
30,0.069209,0.027795,0.043994,0.993238,13032.000000
40,0.029981,0.013816,0.021519,0.996249,17377.000000
50,0.013246,0.012072,0.016893,0.996487,21714.000000
60,0.009236,0.005255,0.010731,0.998490,26070.000000
70,0.007204,0.006028,0.009138,0.997740,30452.000000
80,0.003028,0.005734,0.007674,0.997740,34813.000000
90,0.011635,0.004702,0.007529,0.997990,39116.000000
100,0.008894,0.004479,0.007536,0.997990,43449.000000


{'Тривалість, хв': 5.409285926818848,
 'Пікова памʼять GPU, GB': 3.664846336,
 'Validation loss': 0.004478933289647102}

## 9. Підсумкове порівняння

In [25]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA,Qwen3-0.6B повне,Qwen3-4B QLoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00,99.40,99.90
"Точність intent, %",25.6,98.00,57.4,96.90,72.5,97.60,95.90,96.90
"Тривалість, хв",NaN,2.50,NaN,2.56,NaN,3.81,1.66,5.41
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75,10.80,3.66
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00,0.01,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19,NaN,NaN


**Що варто помітити:**
- LoRA тренує близько 1–2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й памʼяті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і памʼяттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після — лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше памʼяті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

22. ***Як довго виконувалися клітинки ноутбуку 03_sft_lora? Чи були проблеми з запуском, які?***\
Комірки з донавчанням/тестуванням виконувались досить довго (5-15 хв), але проблем з запуском не було.
23. ***Прокоментуйте вивід "Памʼять GPU після завантаження". Чи очікуваним для Вас є результат?***\
Цей вивід виводить пам'ять, яку займає модель після завантаження, 1.2Гб для 0.6B, 3.5Гб для 1.7B та 8.1Гб для 4B. Цей результат є очікуваним, адже для моделі у fp16 очікуваним є 2 байти на параметр.
24. ****Що робить рядок with torch.no_grad() у функції classify()?**\
У функції classify() немає такого рядка. У функції classify_batch() цей рядок використовують для того, щоб не обраховувати та не зберігати градієнти у блоці кода, адже вони не потрібні для передбачення.
25. ***Для чого у classify() задано do_sample=False?***\
У classify_batch(). Для того, щоб модель завжди обирала найімовірніший наступний токен, адже нам потрібні точні відповіді зі структурою JSON'ів.
26. ***В демо 1 було натреновано класифікатор з базової моделі за допомогою заміни голови. В демо 3 в ході інференсу моделі також відбувається класифікація звернень. Чому не можна застосувати такий же підхід?***\
Вихідні дані у демо 3 більш складні, адже складаються з двох полів категорія + намір з великою к-стю можливих варіантів, тоді для нової голови прийшлось би створювати нейрон для кожної комбінації категорія-намір або дві голови. Також, у випадку з новою головою - вона має фіксовану кількість нейроннів на останньому шарі і якщо буде потреба додати нову категорію, то архітектуру доведеться фізично змінювати для додавання нових вихідних нейронів, в тей час коли для SFT додати нову категорію буде простіше за допомогою правильного промпту або невеликого донавчання. Також з SFT модель зберігла вміння генерувати текст.
27. ***Порівняйте кількість токенів короткого й детального промпту (розділ 2). Як ця різниця вплине на вартість обробки 10 000 звернень на день?***\
Довжина короткого промпту - 11 токенів, а довгого - 166. Різниця 15-кратна. Це зайві 1.5млн токенів на день при 10000 звернень/день. Враховуючи, що довжина всього діалогу з коротким промптом близько 100 токенів, тоді загальна різниця в токенах та ціні буде в 2-2.5 рази.
28. ***Для чого в датасеті задано колонку chat_template_kwargs з enable_thinking=False? Що, на Вашу думку, сталося б із моделлю, якби її донавчали на цих прикладах у режимі /think?***\
Цю колонку задано, щоб модель відповідала без міркування. Якби її увімкнули, модель могла б розівчитись міркувати, адже в датасеті відсутні міркування.
29. ***Що означає параметр assistant_only_loss=True? На яких токенах модель вчилася б без нього і чим це погано для нашої задачі?***\
Що loss рахується тільки на токенах відповіді. Інакше б модель би також вчилась на токенах системного промпту та промпту користувача і намагалась б передбачувати їх. Це могло б призвести до галюцинацій під час роботи(коли модель намагається продивжити запит користувача, замість відповіді) і до витрати ресурсів моделі під час навчання на ці передбачування, замість пошуку логічних зв'язків між запитом та відповіддю.
30. ***Порівняйте LoRA і QLoRA за пам'яттю, часом тренування і якістю (розділ 9). У яких випадках Ви обрали б QLoRA?***\
QLoRA потребувала більш ніж в 2 рази менше пам'яті для донавчання, ніж LoRA, але потрібно було майже в 2 рази більше часу. Якість фінальних моделей була досить схожа, з невеликою перевагою у LoRA. Я б обирав QLoRA для тих випадків, коли кількості VRAM (або бюджетів) не вистачає для LoRA і обрання іншої моделі не є варіантом.